# DuckDB - OneLake Files

In this demo we attempt to connect to remove files.

In [54]:
import duckdb

import os

from azure.storage.filedatalake import (
    DataLakeServiceClient,
    DataLakeDirectoryClient,
    FileSystemClient
)
from azure.identity import DefaultAzureCredential


In [55]:
ACCOUNT_NAME = "onelake"
WORKSPACE_NAME = "DuckDbPolarsDemos"
DATA_PATH = "PricePaidData.Lakehouse/Files/land_registry/"
FILE = "pp-2023.csv"

In [56]:
account_url = f"https://{ACCOUNT_NAME}.dfs.fabric.microsoft.com"
abfss_path = f"abfss://{WORKSPACE_NAME}@{ACCOUNT_NAME}.dfs.fabric.microsoft.com/{DATA_PATH}{FILE}"

## Conventional connection to Onelake

In [57]:
token_credential = DefaultAzureCredential()
service_client = DataLakeServiceClient(account_url, credential=token_credential)

#Create a file system client for the workspace
file_system_client = service_client.get_file_system_client(WORKSPACE_NAME)

#List a directory within the filesystem
paths = file_system_client.get_paths(path=DATA_PATH)

In [58]:
token = token_credential.get_token("https://storage.azure.com/.default")

In [59]:
file_system_client.get_file_system_properties

<bound method FileSystemClient.get_file_system_properties of <azure.storage.filedatalake._file_system_client.FileSystemClient object at 0x7f412a596f00>>

In [60]:
for path in paths:
    print(path)

{'name': 'PricePaidData.Lakehouse/Files/land_registry/pp-2000.csv', 'owner': '954b20be-abdc-4fbc-9e27-545547e1c715', 'group': '954b20be-abdc-4fbc-9e27-545547e1c715', 'permissions': 'rw-r-----', 'last_modified': datetime.datetime(2025, 4, 26, 16, 29, 11), 'is_directory': False, 'etag': '0x8DD84DF7A1692F8', 'content_length': 193369527, 'creation_time': datetime.datetime(2025, 4, 26, 16, 29, 8, 75418, tzinfo=datetime.timezone.utc), 'expiry_time': None, 'encryption_scope': None, 'encryption_context': None}
{'name': 'PricePaidData.Lakehouse/Files/land_registry/pp-2001.csv', 'owner': '954b20be-abdc-4fbc-9e27-545547e1c715', 'group': '954b20be-abdc-4fbc-9e27-545547e1c715', 'permissions': 'rw-r-----', 'last_modified': datetime.datetime(2025, 4, 26, 16, 29, 11), 'is_directory': False, 'etag': '0x8DD84DF7A3A8DB4', 'content_length': 220234122, 'creation_time': datetime.datetime(2025, 4, 26, 16, 29, 8, 14016, tzinfo=datetime.timezone.utc), 'expiry_time': None, 'encryption_scope': None, 'encryption_

In [61]:
os.environ["AZURE_LOG_LEVEL"] = "verbose"

In [62]:
duckdb_connection = duckdb.connect(database=":memory:", read_only=False)

In [63]:
duckdb_connection.sql(
    """
    INSTALL azure;
    LOAD azure;
    """
)

In [ ]:
duckdb_connection.sql(f""" CREATE or replace PERSISTENT SECRET onelake ( TYPE AZURE, PROVIDER ACCESS_TOKEN, ACCESS_TOKEN '{token.token}', ACCOUNT_NAME 'onelake')""")

┌─────────┐
│ Success │
│ boolean │
├─────────┤
│ true    │
└─────────┘

In [ ]:
# duckdb_connection.sql("CREATE or replace PERSISTENT SECRET onelake (TYPE azure,PROVIDER credential_chain, CHAIN 'cli',ACCOUNT_NAME 'onelake');")

In [69]:
duckdb_connection.sql(
    f"""
    SELECT
    column00 AS 'id',
    column01 AS 'price',
    column02 AS 'date',
    column03 AS 'postcode',
    column04 AS 'property_type',
    column05 AS 'old_new',
    column06 AS 'duration',
    column07 AS 'paon',
    column08 AS 'saon',
    column09 AS 'street',
    column10 AS 'locale',
    column11 AS 'town_city',
    column12 AS 'district',
    column13 AS 'county',
    column14 AS 'ppd_category',
    column15 AS 'record_type',
    year(date) AS 'year_of_sale',
    month(date) AS 'month_of_sale'
    FROM read_csv(
        '{abfss_path}',
        header=False
    );
    """
)

[2025-04-26T20:12:50.5950413Z T: 139918628726592] INFO  : HTTP Request : HEAD https://onelake.blob.fabric.microsoft.com/DuckDbPolarsDemos/PricePaidData.Lakehouse/Files/land_registry/pp-2023.csv
authorization : REDACTED
user-agent : azsdk-cpp-storage-blobs/12.13.0 (Linux 5.15.153.1-microsoft-standard-WSL2 x86_64 #1 SMP Fri Mar 29 23:14:13 UTC 2024 Cpp/201402)
x-ms-client-request-id : 63014f6a-26ce-4bda-bbd9-e99554b94683
x-ms-date : Sat, 26 Apr 2025 20:12:50 GMT
x-ms-version : 2024-08-04
[2025-04-26T20:12:50.5950548Z T: 139918628726592] DEBUG : [CURL Transport Adapter]: Creating a new session.
[2025-04-26T20:12:50.5950611Z T: 139918628726592] DEBUG : [CURL Transport Adapter]: Spawn new connection.
[2025-04-26T20:12:50.7632499Z T: 139918628726592] WARN  : HTTP Transport error: Fail to get a new connection for: https://onelake.blob.fabric.microsoft.com. Problem with the SSL CA cert (path? access rights?)
[2025-04-26T20:12:50.7632683Z T: 139918628726592] INFO  : HTTP Retry attempt #1 will b

IOException: IO Error: AzureBlobStorageFileSystem could not open file: 'abfss://DuckDbPolarsDemos@onelake.dfs.fabric.microsoft.com/PricePaidData.Lakehouse/Files/land_registry/pp-2023.csv', unknown error occurred, this could mean the credentials used were wrong. Original error message: 'Fail to get a new connection for: https://onelake.blob.fabric.microsoft.com. Problem with the SSL CA cert (path? access rights?)' 